In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
from src.boundary import BCConfig
from src.solvers import ExplicitEuler2D, CrankNicolson2D
from src.analytics import analytical_solution_sin, l2_error, relative_l2_error
from src.reaction import ReactionDiffusionSolver, ReactionType
from src.visualization import plot_heatmap, animate_diffusion
%matplotlib inline

## 2. 2D Heat Equation: Explicit Euler vs Crank-Nicolson

Initial condition: $u(x,y,0) = \sin(\pi x)\sin(\pi y)$ with zero Dirichlet BCs.
Analytical solution: $u(x,y,t) = \sin(\pi x)\sin(\pi y)e^{-2\alpha\pi^2 t}$.

In [ ]:
def sin_ic(X, Y):
    return np.sin(np.pi * X) * np.sin(np.pi * Y)

bc = BCConfig.dirichlet_all(0.0)

solver_euler = ExplicitEuler2D(nx=50, ny=50, Lx=1.0, Ly=1.0, alpha=0.01, bc=bc)
solver_cn = CrankNicolson2D(nx=50, ny=50, Lx=1.0, Ly=1.0, alpha=0.01, bc=bc)

dt_small = 0.001
r_euler = solver_euler.solve(sin_ic, dt=dt_small, t_total=0.1)
r_cn = solver_cn.solve(sin_ic, dt=dt_small, t_total=0.1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
X, Y = np.meshgrid(r_euler.grid_x, r_euler.grid_y, indexing='ij')
u_exact = analytical_solution_sin(X, Y, t=0.1, alpha=0.01)
plot_heatmap(r_euler.u, r_euler.grid_x, r_euler.grid_y, title='Euler (dt=0.001)', ax=ax1)
plot_heatmap(r_cn.u, r_cn.grid_x, r_cn.grid_y, title='Crank-Nicolson (dt=0.001)', ax=ax2)
fig.tight_layout()

err_e = l2_error(r_euler.u, u_exact)
err_cn = l2_error(r_cn.u, u_exact)
print(f'Euler L2 error: {err_e:.6e}')
print(f'CN L2 error:    {err_cn:.6e}')

## 3. Stability Comparison

Crank-Nicolson is unconditionally stable. Euler is conditionally stable (CFL: $r_x + r_y \le 0.5$).

In [ ]:
np.random.seed(42)
def random_ic(X, Y):
    return np.random.random(X.shape) * 0.1

dt_big = 0.04  # > CFL for 30x30 grid

s_e_big = ExplicitEuler2D(nx=30, ny=30, alpha=0.01, bc=bc)
s_cn_big = CrankNicolson2D(nx=30, ny=30, alpha=0.01, bc=bc)

try:
    r_e_big = s_e_big.solve(random_ic, dt=dt_big, t_total=0.5)
    stable_e = r_e_big.u.max() < 10
except Exception:
    stable_e = False

r_cn_big = s_cn_big.solve(random_ic, dt=dt_big, t_total=0.5)

print(f'Euler dt={dt_big} (beyond CFL): {"BLEW UP" if not stable_e else "stable"} (max={r_e_big.u.max():.2f})')
print(f'CN dt={dt_big} (beyond CFL):     stable (max={r_cn_big.u.max():.4f})')

## 4. Diffusion Animation

In [ ]:
rc = solver_cn.solve(sin_ic, dt=0.002, t_total=0.2, record_every=5)
anim = animate_diffusion(rc.u_history, rc.grid_x, rc.grid_y, rc.times, 
                          title='Crank-Nicolson', interval=80)
plt.close()  # prevent duplicate display
HTML(anim.to_jshtml())

## 5. Fisher-KPP Reaction-Diffusion

$u_t = D\nabla^2 u + r u(1-u)$ — a traveling wave front.

In [ ]:
def circle_ic(X, Y):
    r = np.sqrt((X-1.0)**2 + (Y-1.0)**2)
    return (r < 0.2).astype(float)

fk = ReactionDiffusionSolver(nx=60, ny=60, Lx=2.0, Ly=2.0, D=0.005, r=1.0)
r_fk = fk.solve(circle_ic, t_total=3.0, dt=0.02)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, t_idx in enumerate([0, len(r_fk['times'])//2, -1]):
    plot_heatmap(r_fk['u_history'][t_idx], r_fk['grid_x'], r_fk['grid_y'],
                 title=f't={r_fk["times"][t_idx]:.2f}', ax=axes[i])
fig.tight_layout()

## 6. Gray-Scott Pattern Formation

$u_t = D_u\nabla^2 u - uv^2 + F(1-u)$, $v_t = D_v\nabla^2 v + uv^2 - (F+k)v$

In [ ]:
def gs_u_ic(X, Y):
    u = np.ones(X.shape)
    u[40:45, 40:45] = 0.5
    return u

def gs_v_ic(X, Y):
    v = np.zeros(X.shape)
    v[40:45, 40:45] = 0.25
    return v

gs = ReactionDiffusionSolver(nx=80, ny=80, Lx=2.0, Ly=2.0,
                              reaction_type=ReactionType.GRAY_SCOTT,
                              F=0.04, k=0.06, Du=0.001, Dv=0.0005)
r_gs = gs.solve(gs_u_ic, t_total=20.0, dt=0.1, ic_func_v=gs_v_ic)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plot_heatmap(r_gs['u'], r_gs['grid_x'], r_gs['grid_y'], title='Gray-Scott: u', ax=ax1)
plot_heatmap(r_gs['v'], r_gs['grid_x'], r_gs['grid_y'], title='Gray-Scott: v', ax=ax2,
             cmap='viridis')
fig.tight_layout()

---
Built with [uv](https://docs.astral.sh/uv/), NumPy, SciPy, Matplotlib.